In [39]:
# from astropy.visualization import hist
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Locate 02_Model when running from anywhere inside this repository.
working_directory = Path.cwd().resolve()
model_root = next(
    parent / "02_Model"
    for parent in (working_directory, *working_directory.parents)
    if (parent / "02_Model" / "v3" / "src").is_dir()
)

if str(model_root) not in sys.path:
    sys.path.insert(0, str(model_root))

from v3.src import create_mines, params, units

data_directory = model_root / "v3" / "data"
raw_data = pd.read_csv(
    data_directory / "master_database_projects_UPDATED.csv"
)

import importlib
importlib.reload(create_mines)

def add_lce_columns(df):
    """Reporting columns on an LCE basis, derived from the Li columns in the dataset."""
    df = df.copy()
    df["capacity_recovered_lce"] = df["capacity_recovered_li"] * units.LCE_PER_LI
    df["opex_per_t_lce"] = df["opex_per_t_li"] / units.LCE_PER_LI
    df["project_cost_per_t_lce"] = df["project_cost_per_t_li"] / units.LCE_PER_LI
    return df


mines = add_lce_columns(create_mines.create_mines_dataset(raw_data, seed=params.INITIALISATION_SEED, base_year=params.BASE_YEAR, 
historical_utilisation=params.HISTORICAL_UTILISATION))


In [40]:
from v3.src.setup_model import create_mine_assets

mine_assets = create_mine_assets(mines)

# Illustrative full-lifetime output at 100% utilisation

In [41]:
mines["assumed_lifetime_output_t_lce"] = mines["capacity_recovered_lce"] * mines["lifespan_years"]

mines["comparison_cost_per_t_lce"] = mines["opex_per_t_lce"] + mines["capex_total"] / mines["assumed_lifetime_output_t_lce"]

## Create chart function

In [42]:
STAGE_COLOURS = {
    "operation": "#2166ac",
    "construction": "#e69f00",
    "pre-construction": "#999999",
}

ROUTE_PATTERNS = {
    "hard_rock": "",
    "brine": "/",
}

def cost_curve(data, cost_column, title):
    curve = data.sort_values([cost_column, "asset_id"]).copy()

    if curve.empty:
        raise ValueError("No projects selected.")

    quantities = curve[
        [cost_column, "capacity_recovered_lce"]
    ].to_numpy(dtype=float)

    if not np.isfinite(quantities).all() or (quantities <= 0).any():
        raise ValueError("Costs and capacities must be finite and positive.")

    curve["width_kt"] = (
        curve["capacity_recovered_lce"].astype(float) / 1000
    )
    curve["right_kt"] = curve["width_kt"].cumsum()
    curve["centre_kt"] = curve["right_kt"] - curve["width_kt"] / 2

    fig = go.Figure()

    for (stage, route), group in curve.groupby(
        ["stage_at_base_year", "deposit_type"]
    ):
        fig.add_trace(
            go.Bar(
                x=group["centre_kt"],
                y=group[cost_column],
                width=group["width_kt"],
                name=f"{stage} | {route}",
                marker={
                    "color": STAGE_COLOURS[stage],
                    "pattern": {"shape": ROUTE_PATTERNS[route]},
                    "opacity": np.where(
                        group["reserves_imputed"], 0.4, 1.0
                    ).tolist(),
                    "line": {"width": 0},
                },
                customdata=group[
                    [
                        "name",                       # 0
                        "asset_id",                   # 1
                        "capacity_recovered_lce",     # 2
                        "reserves_imputed",            # 3
                        "deposit_type_imputed",        # 4
                        "lifespan_years_imputed",      # 5
                        "reserves_filled",             # 6
                        "capacity_years",              # 7
                    ]
                ].to_numpy(),

                hovertemplate=(
                    "%{customdata[0]} (ID %{customdata[1]})"
                    "<br>Cost: %{y:,.0f} USD/t intermediate LCE"
                    "<br>Capacity: %{customdata[2]:,.0f} t intermediate LCE/year"
                    "<br>Reference stock: %{customdata[6]:,.0f} t contained LCE"
                    "<br>Capacity-inference duration: %{customdata[7]:.1f} years"
                    "<br>Stock imputed: %{customdata[3]}"
                    "<br>Deposit type imputed: %{customdata[4]}"
                    "<br>Lifespan imputed: %{customdata[5]}"
                    "<extra>%{fullData.name}</extra>"
                ),
                            )
                        )

    fig.update_layout(
        title=title,
        template="plotly_white",
        barmode="overlay",
        bargap=0,
        xaxis_title=(
            "Cumulative modelled capacity (kt intermediate LCE/year)"
        ),
        yaxis_title="2024 USD/t intermediate LCE",
        legend_title_text="Stage | route",
        height=500,
    )

    fig.update_xaxes(range=[0, curve["right_kt"].iloc[-1]])
    fig.update_yaxes(rangemode="tozero")

    return fig

# Run multiple seeds

In [43]:
importlib.reload(create_mines)

seed_datasets = {
    seed: add_lce_columns(create_mines.create_mines_dataset(
        raw_data,
        seed=seed,
        base_year=params.BASE_YEAR,
        historical_utilisation=params.HISTORICAL_UTILISATION,
    ))
    for seed in range(1, 41)
}

In [44]:
def plot_seed_curves(datasets, reported_only=False):
    fig = go.Figure()
    curves = {}

    for seed, data in datasets.items():
        selected = data["stage_at_base_year"].isin(
            ["operation", "construction", "pre-construction"]
        )

        if reported_only:
            selected &= ~data["reserves_imputed"]

        curve = data.loc[selected].copy()

        curve["comparison_cost_per_t_lce"] = (
            curve["opex_per_t_lce"].astype(float)
            + curve["capex_total"].astype(float)
            / (
                curve["capacity_recovered_lce"].astype(float)
                * curve["lifespan_years"].astype(float)
            )
        )

        curve = curve.sort_values(
            ["comparison_cost_per_t_lce", "asset_id"]
        ).reset_index(drop=True)

        capacity_kt = (
            curve["capacity_recovered_lce"].to_numpy(dtype=float)
            / 1000
        )
        costs = curve["comparison_cost_per_t_lce"].to_numpy(
            dtype=float
        )

        if (
            curve.empty
            or not np.isfinite(capacity_kt).all()
            or not np.isfinite(costs).all()
            or (capacity_kt <= 0).any()
            or (costs <= 0).any()
        ):
            raise ValueError(f"Invalid curve for seed {seed}")

        right_edges = capacity_kt.cumsum()
        left_edges = np.r_[0, right_edges[:-1]]

        curve["cumulative_capacity_kt"] = right_edges
        curves[seed] = curve

        fig.add_trace(
            go.Scatter(
                x=np.column_stack(
                    [left_edges, right_edges]
                ).ravel(),
                y=np.repeat(costs, 2),
                mode="lines",
                name=f"Seed {seed}",
                line={
                    "color": "rgba(33, 102, 172, 0.25)",
                    "width": 1,
                },
                showlegend=False,
                hovertemplate=(
                    f"Seed {seed}"
                    "<br>Capacity: %{x:,.1f} kt intermediate LCE/year"
                    "<br>Cost: %{y:,.0f} USD/t intermediate LCE"
                    "<extra></extra>"
                ),
            )
        )

    subset = (
        "Reported stocks only"
        if reported_only
        else "Reported and imputed stocks"
    )

    fig.update_layout(
        title=f"{subset} — {len(datasets)} initialisation seeds",
        template="plotly_white",
        xaxis_title=(
            "Cumulative potential capacity "
            "(kt intermediate LCE/year)"
        ),
        yaxis_title=(
            "OPEX + initial CAPEX per lifetime tonne"
            "<br>(2024 USD/t intermediate LCE)"
        ),
        height=550,
    )
    fig.update_xaxes(rangemode="tozero")
    fig.update_yaxes(rangemode="tozero")

    return fig, curves

In [45]:
operating = mines.loc[
    mines["stage_at_base_year"].eq("operation")
]

cost_curve(
    operating,
    "opex_per_t_lce",
    "Operating mines — mining OPEX",
).show()

In [46]:
prospective = mines.loc[
    mines["stage_at_base_year"].eq("pre-construction")
]

cost_curve(
    prospective,
    "project_cost_per_t_lce",
    "Prospective projects — OPEX plus initial CAPEX",
).show()

In [47]:
active_and_prospective = mines.loc[
    mines["stage_at_base_year"].isin(STAGE_COLOURS)
]

cost_curve(
    active_and_prospective,
    "comparison_cost_per_t_lce",
    "All stages — OPEX plus initial CAPEX per assumed lifetime tonne",
).show()

In [48]:
reported_stock_projects = active_and_prospective.loc[
    ~active_and_prospective["reserves_imputed"]
]

cost_curve(
    reported_stock_projects,
    "comparison_cost_per_t_lce",
    "Reported-stock subset — OPEX plus initial CAPEX "
    "per assumed lifetime tonne",
).show()

# Repeat the graphs across multiple runs

In [49]:
all_seeds_fig, all_seed_curves = plot_seed_curves(
    seed_datasets,
    reported_only=False,
)
all_seeds_fig.show()

In [50]:
reported_seeds_fig, reported_seed_curves = plot_seed_curves(
    seed_datasets,
    reported_only=True,
)
reported_seeds_fig.show()